In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

In [2]:
data = pd.read_csv("student_habits_performance.csv")

In [3]:
data.head()

,student_id,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,S1000,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,S1001,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,S1002,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,S1003,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,S1004,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   student_id                     1000 non-null   object 
 1   age                            1000 non-null   int64  
 2   gender                         1000 non-null   object 
 3   study_hours_per_day            1000 non-null   float64
 4   social_media_hours             1000 non-null   float64
 5   netflix_hours                  1000 non-null   float64
 6   part_time_job                  1000 non-null   object 
 7   attendance_percentage          1000 non-null   float64
 8   sleep_hours                    1000 non-null   float64
 9   diet_quality                   1000 non-null   object 
 10  exercise_frequency             1000 non-null   int64  
 11  parental_education_level       909 non-null    object 
 12  internet_quality               1000 non-null   ob

In [5]:
features = [

    "study_hours_per_day",
    "mental_health_rating",
    "exercise_frequency",
    "sleep_hours",

    "social_media_hours",
    "netflix_hours",

    "attendance_percentage",

    "internet_quality",
    "extracurricular_participation"

]


df = data[features + ["exam_score"]].copy()

In [8]:
df["distraction_hours"] = (
    df["social_media_hours"]
    +
    df["netflix_hours"]
)

In [9]:
df["study_vs_distraction"] = (
    df["study_hours_per_day"]
    -
    df["distraction_hours"]
)

In [10]:
df["study_mental_interaction"] = (
    df["study_hours_per_day"]
    *
    df["mental_health_rating"]
)

In [11]:
cat_cols = df.select_dtypes(include="object").columns
encoders = {}


for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

In [12]:
X = df.drop("exam_score",axis=1)

y = df["exam_score"]

In [13]:
X_train, X_test, y_train, y_test = train_test_split( X,y,test_size=0.2,random_state=42)

In [14]:
model = XGBRegressor(

    objective="reg:squarederror",

    n_estimators=1500,

    learning_rate=0.02,

    max_depth=5,

    min_child_weight=2,

    subsample=0.9,

    colsample_bytree=0.9,

    reg_alpha=0.1,

    reg_lambda=1.5,

    random_state=42

)

In [15]:
model.fit(X_train,y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.9
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [16]:
pred = model.predict(X_test)


mae = mean_absolute_error(
    y_test,
    pred
)


rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred
    )
)


r2 = r2_score(
    y_test,
    pred
)

In [17]:
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

MAE  : 4.57
RMSE : 5.64
R²   : 0.8759


In [18]:
importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance": model.feature_importances_

})


print("\nFeature Importance")

display(
    importance.sort_values(
        by="Importance",
        ascending=False
    )
)


Feature Importance


,Feature,Importance
0,study_hours_per_day,0.530856
11,study_mental_interaction,0.168975
10,study_vs_distraction,0.127715
2,exercise_frequency,0.041879
1,mental_health_rating,0.031295
3,sleep_hours,0.027632
9,distraction_hours,0.022469
6,attendance_percentage,0.013838
5,netflix_hours,0.010887
4,social_media_hours,0.010830


In [19]:
joblib.dump(
    model,
    "xgb.pkl"
)


joblib.dump(
    encoders,
    "encoders.pkl"
)


joblib.dump(
    list(X.columns),
    "features.pkl"
)

print("MODEL SAVED SUCCESSFULLY")

MODEL SAVED SUCCESSFULLY
